# Exploratory Data Analysis (EDA) & Cross-Table Correlation
This notebook contains a complete exploratory data analysis (EDA) of the following database tables:
1. **`tbl_users`**: User records, roles, statuses, and coordinates.
2. **`customers`**: The customer profiles, implementation types, salespersons, and account owners (excluding the legacy `tbl_customer` table).
3. **`tbl_customer_services_beta`**: Operational service logs, request volumes, coordinators, and temporal trends.

Furthermore, we analyze how each table is correlated to the others via key columns.

## 1. Database Connection and Configuration
We connect to the MySQL database using SQLAlchemy and load the tables into Pandas DataFrames.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import urllib.parse

# Set style for plots
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Connection config
DB_USER = "feroz"
DB_PASSWORD = urllib.parse.quote_plus("feroz@1289")
DB_HOST = "85.195.89.47"
DB_PORT = "3306"
DB_NAME = "admin_latest_ai_test"

connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)
print(f"✓ Connected successfully to database: {DB_NAME}")

df_users = pd.read_sql_table('tbl_users', engine)
print(f"✓ Loaded tbl_users ({df_users.shape[0]} rows)")

df_customers = pd.read_sql_table('customers', engine)
print(f"✓ Loaded customers ({df_customers.shape[0]} rows)")

df_services = pd.read_sql_table('tbl_customer_services_beta', engine)
print(f"✓ Loaded tbl_customer_services_beta ({df_services.shape[0]} rows)")

## 2. Analysis of `tbl_users` Table
The `tbl_users` table contains the details of active staff, administrators, development teams, and support roles.

In [ ]:
print("=== Users Table Information ===")
print(f"Shape: {df_users.shape}")
print(f"Columns: {list(df_users.columns)}\n")

print("=== User Types Breakdown ===")
print(df_users['user_type'].value_counts())

print("\n=== User Status Breakdown ===")
print(df_users['user_status'].value_counts())

In [ ]:
# Plotting the distribution of user types
plt.figure(figsize=(10, 5))
sns.countplot(y='user_type', data=df_users, order=df_users['user_type'].value_counts().index, palette='viridis')
plt.title('Distribution of User Roles in tbl_users')
plt.xlabel('Count')
plt.ylabel('User Type')
plt.tight_layout()
plt.show()

## 3. Analysis of `customers` Table (excluding legacy `tbl_customer`)
We inspect the active customer profiles, implementation types, salespersons, and account owners.

In [ ]:
print("=== Customers Table Information ===")
print(f"Shape: {df_customers.shape}")
print(f"Columns: {list(df_customers.columns[:15])}...\n")

print("=== Implementation Types in Customers Table ===")
print(df_customers['implementationType'].value_counts().head(10))

In [ ]:
# Plot top implementation types
plt.figure(figsize=(10, 5))
sns.countplot(y='implementationType', data=df_customers, order=df_customers['implementationType'].value_counts().head(10).index, palette='rocket')
plt.title('Top 10 Customer Implementation Types')
plt.xlabel('Count')
plt.ylabel('Implementation Type')
plt.tight_layout()
plt.show()

## 4. Analysis of `tbl_customer_services_beta` Table
This table stores the service requests and logs.

In [ ]:
print("=== Services Table Information ===")
print(f"Shape: {df_services.shape}")
print(f"Columns: {list(df_services.columns)}\n")

print("=== Service Request Status Breakdown ===")
print(df_services['customer_service_status'].value_counts())

## 5. Cross-Table Correlation Analysis
Here we study the correlations and relationships between the tables using their shared columns.

In [ ]:
# 1. Correlation: Customers <-> Services
# customer_service_customer_id in df_services matches id in df_customers
df_services_cleaned = df_services[df_services['customer_service_customer_id'] > 0]
services_per_customer = df_services_cleaned['customer_service_customer_id'].value_counts()

print("=== Customer <-> Services Correlation ===")
print(f"Total unique customers in 'customers' table: {df_customers['id'].nunique()}")
print(f"Customers with at least one service ticket: {services_per_customer.nunique()}")
print(f"Top customer by number of service requests:")
top_cust_id = services_per_customer.index[0]
top_cust_name = df_customers[df_customers['id'] == top_cust_id]['name'].values
print(f"Customer ID: {top_cust_id}, Name: {top_cust_name[0] if len(top_cust_name) > 0 else 'Unknown'}, Count: {services_per_customer.iloc[0]}")

In [ ]:
# 2. Correlation: Users <-> Services
# customer_service_created_by, requested_by, and customer_service_L2_assigned_to in services match user_id in tbl_users
print("=== Users <-> Services Correlation ===")
# Convert columns to match user_id type (int or numeric)
df_services['created_by_int'] = pd.to_numeric(df_services['customer_service_created_by'], errors='coerce')
df_services['requested_by_int'] = pd.to_numeric(df_services['requested_by'], errors='coerce')
df_services['assigned_to_int'] = pd.to_numeric(df_services['customer_service_L2_assigned_to'], errors='coerce')

# Map user IDs to names
user_id_to_name = df_users.set_index('user_id')['user_name'].to_dict()

df_services['creator_name'] = df_services['created_by_int'].map(user_id_to_name)
df_services['requester_name'] = df_services['requested_by_int'].map(user_id_to_name)
df_services['assignee_name'] = df_services['assigned_to_int'].map(user_id_to_name)

print("Top 5 Creators of Service Tickets:")
print(df_services['creator_name'].value_counts().head(5))
print("\nTop 5 Requesters of Service Tickets:")
print(df_services['requester_name'].value_counts().head(5))
print("\nTop 5 Assignees (L2 Assigned To) of Service Tickets:")
print(df_services['assignee_name'].value_counts().head(5))

In [ ]:
# 3. Correlation: Users <-> Customers
# salesPerson and accountOwner in customers table match user_id in tbl_users
print("=== Users <-> Customers Correlation ===")
df_customers['salesperson_int'] = pd.to_numeric(df_customers['salesPerson'], errors='coerce')
df_customers['accountowner_int'] = pd.to_numeric(df_customers['accountOwner'], errors='coerce')

df_customers['salesperson_name'] = df_customers['salesperson_int'].map(user_id_to_name)
df_customers['accountowner_name'] = df_customers['accountowner_int'].map(user_id_to_name)

print("Top 5 Sales Persons managing Customers:")
print(df_customers['salesperson_name'].value_counts().head(5))
print("\nTop 5 Account Owners managing Customers:")
print(df_customers['accountowner_name'].value_counts().head(5))

In [ ]:
# Visualizing user-to-service relationship
plt.figure(figsize=(10, 5))
sns.countplot(y='assignee_name', data=df_services, order=df_services['assignee_name'].value_counts().head(10).index, palette='mako')
plt.title('Top 10 Service Ticket Assignees by Name')
plt.xlabel('Ticket Count')
plt.ylabel('Assignee Name')
plt.tight_layout()
plt.show()